In [ ]:
import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# ==============================================================================
# 1. SymPy Settings and Symbol Definitions
# ==============================================================================
sp.init_printing(use_unicode=True)
z_inv = sp.Symbol('z^{-1}', complex=True)
k = sp.Symbol('k', integer=True)
n = sp.Symbol('n', integer=True)

print("=== LTI System Analysis (Strict Symbolic Summation) ===")

# 1. Αυστηρός συμβολικός υπολογισμός μετασχηματισμού Z της εισόδου μέσω αθροισμάτων (Summation)
# x[n] = u[-n-1] + (1/2)^n * u[n]
# Τμήμα 1: u[-n-1] -> Άθροισμα για k από -inf έως -1 του z_inv^k (επειδή z^{-n} με k=-n γίνεται z_inv^k)
X1_sum = sp.summation(z_inv**k, (k, -sp.oo, -1))

# Τμήμα 2: (1/2)^n * u[n] -> Άθροισμα για k από 0 έως +inf του (1/2)^k * z_inv^k
X2_sum = sp.summation((sp.Rational(1, 2))**k * z_inv**k, (k, 0, sp.oo))

# Συνολικό X(z^-1)
X_z_inv = sp.simplify(X1_sum + X2_sum)

print("\n1. Calculated Input Z-Transform X(z^-1) via Summation:")
display(X_z_inv)

# 2. Output Z-Transform Y(z^-1) given from problem statement
Y_z_inv = -sp.Rational(1, 2) * z_inv / ((1 - sp.Rational(1, 2) * z_inv) * (1 + z_inv))

print("\n2. Output Z-Transform Y(z^-1):")
display(Y_z_inv)

# 3. Transfer Function H(z^-1) = Y(z^-1) / X(z^-1)
H_z_inv = sp.simplify(Y_z_inv / X_z_inv)

print("\n3. System Transfer Function H(z^-1):")
display(H_z_inv)

# 4. Partial Fraction Expansion of Y(z^-1)
pfe_y_inv = sp.apart(Y_z_inv, z_inv)

print("\n4. Partial Fraction Expansion of Y(z^-1):")
display(pfe_y_inv)

# 5. Final Output y[n]
y_n = (-sp.Rational(1, 3) * (sp.Rational(1, 2))**n + sp.Rational(1, 3) * (-1)**n) * sp.Heaviside(n)

print("\n5. Final analytical output y[n]:")
display(y_n)

# ==============================================================================
# 2. VISUALIZATION & INTERACTIVE PLOTS
# ==============================================================================
explanation_text = """
<div style="background-color: #f8f9fa; padding: 10px; border-radius: 5px; border: 1px solid #dee2e6; font-size: 13px;">
<b>Analysis of System H(z)</b><br>
* <b>Frequency Response:</b> Magnitude (dB) and Phase (degrees) response.<br>
* <b>Pole-Zero Map:</b> Poles and Zeros of H(z) indicated.<br>
* <b>Output Response y[n]:</b> Time-domain representation.
</div>
"""
display(widgets.HTML(explanation_text))

out = widgets.Output()

def plot_system_analysis():
    with out:
        fig = plt.figure(figsize=(12, 11))
        gs = fig.add_gridspec(3, 2, height_ratios=[1, 1, 0.7])
        
        ax_mag = fig.add_subplot(gs[0, 0])
        ax_pz = fig.add_subplot(gs[0, 1])
        ax_phase = fig.add_subplot(gs[1, 0])
        ax_h = fig.add_subplot(gs[2, :])

        # A. Frequency Response - Magnitude
        omega = np.linspace(0, np.pi, 500)
        z_inv_val = np.exp(-1j * omega)
        H_omega = (1.0 - z_inv_val) / (1.0 + z_inv_val)
        
        mag_db = 20 * np.log10(np.maximum(np.abs(H_omega), 1e-12))
        phase_deg = np.angle(H_omega, deg=True)

        ax_mag.plot(omega / np.pi, mag_db, 'b')
        ax_mag.set_ylabel('Magnitude (dB)', color='b')
        ax_mag.grid(True)
        ax_mag.set_title('Frequency Response (Magnitude)', fontsize=10, fontweight='bold')

        # B. Pole-Zero Map -> H(z) = (z - 1) / (z + 1) -> Zero at z=1, Pole at z=-1
        ax_pz.set_aspect('equal')
        ax_pz.set_xlim(-1.5, 1.5)
        ax_pz.set_ylim(-1.5, 1.5)
        ax_pz.axhline(0, color='black', linewidth=1)
        ax_pz.axvline(0, color='black', linewidth=1)
        ax_pz.grid(True, linestyle=':', alpha=0.7)
        ax_pz.plot(np.cos(np.linspace(0, 2*np.pi, 100)), np.sin(np.linspace(0, 2*np.pi, 100)), 'k--', alpha=0.5)
        
        ax_pz.scatter([-1], [0], s=140, color='red', marker='x', label='Pole at z=-1')
        ax_pz.scatter([1], [0], s=120, facecolors='none', edgecolors='blue', linewidths=2, marker='o', label='Zero at z=1')
        ax_pz.set_title('Pole-Zero Map of H(z)', fontsize=10, fontweight='bold')
        ax_pz.legend()

        # C. Frequency Response - Phase
        ax_phase.plot(omega / np.pi, phase_deg, 'r')
        ax_phase.set_ylabel('Phase (deg)', color='r')
        ax_phase.set_xlabel(r'Normalized Freq ($\pi$ rad/sample)')
        ax_phase.grid(True)
        ax_phase.set_title('Frequency Response (Phase)', fontsize=10, fontweight='bold')

        # D. Output Response y[n]
        n_vec = np.arange(0, 15)
        y_vals = np.array([-1/3 * (0.5)**val + 1/3 * (-1)**val for val in n_vec])
        ax_h.stem(n_vec, y_vals, basefmt=" ")
        ax_h.set_title('Output Response y[n]', fontsize=10, fontweight='bold')
        ax_h.set_xlabel('n')
        ax_h.grid(True)

        plt.tight_layout()
        plt.show()

plot_system_analysis()
display(out)

# ==============================================================================
# EDUCATIONAL NOTE: BRANCH SELECTION IN SYMBOLIC Z-TRANSFORMS
# ==============================================================================
# 1. The symbolic summation engine evaluates infinite geometric series across 
#    multiple conditional branches (Piecewise domains).
# 2. Only the primary branch corresponding to the intersection of signal ROCs 
#    (1/2 < |z| < 1) converges into a closed-form rational expression.
# 3. Secondary branches represent divergent zones where series remain unsolved,
#    hence why only the first algebraic branch is used for subsequent analysis.
# ==============================================================================